# 03 — Modelo de Machine Learning

## Proyecto: HR Process Automation Scanner

El objetivo de este notebook es construir un modelo de Machine Learning capaz de predecir la prioridad de automatización de una unidad operativa de HR.

La variable objetivo será:

`automation_priority`

El problema se plantea como una clasificación multiclase, donde cada unidad operativa puede clasificarse como:

- Alta
- Media
- Baja

El modelo utilizará variables operativas como volumen de casos, tiempo de resolución, riesgo de SLA, escalaciones, complejidad, sistema HR, proceso HR, región y canal interno de contacto.

El objetivo final es construir un modelo que pueda integrarse posteriormente en una aplicación de Streamlit para recomendar prioridades de automatización de forma interactiva.

In [ ]:
# Importar librerías y rutas:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data"
FINAL_DATA_PATH = DATA_PATH / "final"
MODELS_PATH = PROJECT_ROOT / "models"
REPORTS_PATH = PROJECT_ROOT / "reports"

MODELS_PATH.mkdir(parents=True, exist_ok=True)
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

print("Ruta data/final:", FINAL_DATA_PATH)
print("Ruta models:", MODELS_PATH)
print("Ruta reports:", REPORTS_PATH)

## 3.1 Carga del dataset limpio

En esta sección se carga el dataset final corregido generado en el notebook anterior.

Este archivo contiene la matriz de oportunidades de automatización con los canales internos HR ya adaptados al contexto de People Operations.

In [ ]:
dataset_path = FINAL_DATA_PATH / "hr_process_automation_ranking_clean.csv"

df = pd.read_csv(dataset_path)

df.head()

In [ ]:
# Revisar dimensiones y columnas:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

df.columns.tolist()

In [ ]:
# Revisar variable objetivo:
df["automation_priority"].value_counts()

## 3.2 Definición del problema de Machine Learning

El problema se define como una clasificación multiclase.

La variable objetivo es `automation_priority`, que clasifica cada unidad operativa según su prioridad de automatización:

- Alta
- Media
- Baja

El modelo aprenderá a predecir esta prioridad usando variables operativas, categóricas y de rendimiento.

In [ ]:
# Definir variables predictoras y target:
target = "automation_priority"

features = [
    "hr_process_name",
    "hr_system_name",
    "hr_contact_channel",
    "region",
    "total_cases",
    "avg_resolution_time_hours",
    "avg_first_response_time_hours",
    "sla_breach_rate",
    "escalation_rate",
    "avg_complexity_score",
    "avg_satisfaction_score",
    "avg_previous_cases",
    "avg_employee_tenure_months",
    "high_priority_rate",
    "urgent_priority_rate"
]

X = df[features].copy()
y = df[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Revisar tipos de variables:
X.dtypes

In [ ]:
# Separar variables numéricas y categóricas:
categorical_features = X.select_dtypes(include="object").columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Variables categóricas:")
print(categorical_features)

print("\nVariables numéricas:")
print(numeric_features)

## 3.3 División en entrenamiento y prueba

Se divide el dataset en dos conjuntos:

- entrenamiento: para que el modelo aprenda;
- prueba: para evaluar el rendimiento sobre datos no vistos.

Se utiliza estratificación para mantener la proporción de clases Alta, Media y Baja en ambos conjuntos.

In [ ]:
# Train/test split:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
# Ver distribución de clases en train y test:
train_distribution = y_train.value_counts(normalize=True).round(3)
test_distribution = y_test.value_counts(normalize=True).round(3)

class_distribution = pd.DataFrame({
    "train_distribution": train_distribution,
    "test_distribution": test_distribution
})

class_distribution

## 3.4 Preprocesamiento

El dataset contiene variables numéricas y categóricas.

Para preparar los datos:

- las variables numéricas serán escaladas con `StandardScaler`;
- las variables categóricas serán transformadas con `OneHotEncoder`.

Este preprocesamiento se integrará dentro de un pipeline para evitar fugas de datos y mantener un flujo reproducible.

In [ ]:
# Crear preprocesador:
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

preprocessor

## 3.5 Entrenamiento de modelos base

Se probarán varios modelos de clasificación para comparar su rendimiento:

- Regresión logística;
- Árbol de decisión;
- Random Forest;
- Gradient Boosting.

La métrica principal será `f1_macro`, porque permite evaluar el rendimiento promedio entre las tres clases sin favorecer únicamente la clase mayoritaria.

In [ ]:
# Definir modelos a evaluar:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

In [ ]:
# Entrenar y evaluar modelos:
model_results = []

trained_models = {}

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    
    results = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro"),
        "recall_macro": recall_score(y_test, y_pred, average="macro"),
        "f1_macro": f1_score(y_test, y_pred, average="macro")
    }
    
    model_results.append(results)
    trained_models[model_name] = pipeline

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values("f1_macro", ascending=False)
)

model_results_df

## 3.6 Selección del modelo ganador

Después de entrenar varios modelos base, el mejor rendimiento lo obtuvo la Regresión Logística.

Aunque es un modelo simple, en este caso funciona muy bien porque la variable objetivo automation_priority fue construida a partir de una lógica de negocio estructurada y reglas de negocio basadas en variables operativas. Por tanto, el modelo está aprendiendo a replicar una lógica de priorización definida previamente (replica la relación entre las variables operativas y la prioridad de automatización).

Lo primero fue construír una lógica de negocio basada en score. Después entrenar un modelo para aprender esa lógica y permitir que futuras unidades operativas puedan clasificarse automáticamente.

Esto dio como resultado un modelo de decisión asistida basado en reglas de negocio y Machine Learning.

Esto es útil porque permite clasificar nuevas unidades operativas de HR de forma automática y consistente.

In [ ]:
# Seleccionar modelo ganador:
best_model_name = model_results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]

print("Mejor modelo:", best_model_name)

In [ ]:
# Predicciones del modelo ganador:
y_pred = best_model.predict(X_test)

y_pred[:10]

In [ ]:
# Classification report del modelo ganador:
print(classification_report(y_test, y_pred))

In [ ]:
# Matriz de confusión del modelo ganador:
cm = confusion_matrix(y_test, y_pred, labels=best_model.classes_)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=best_model.classes_
)

disp.plot()
plt.title(f"Matriz de confusión — {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
# Métricas finales del modelo ganador:
final_model_metrics = pd.DataFrame({
    "métrica": [
        "Modelo ganador",
        "Accuracy",
        "Precision macro",
        "Recall macro",
        "F1 macro"
    ],
    "valor": [
        best_model_name,
        round(accuracy_score(y_test, y_pred), 4),
        round(precision_score(y_test, y_pred, average="macro"), 4),
        round(recall_score(y_test, y_pred, average="macro"), 4),
        round(f1_score(y_test, y_pred, average="macro"), 4)
    ]
})

final_model_metrics

### Interpretación del modelo ganador

La Regresión Logística obtuvo el mejor rendimiento entre los modelos evaluados.

El resultado indica que el modelo es capaz de clasificar correctamente la prioridad de automatización de la mayoría de las unidades operativas.

Este rendimiento alto se explica porque la variable objetivo `automation_priority` fue construida a partir de una lógica de negocio basada en variables operativas como volumen, SLA, escalaciones, complejidad, tiempos y prioridad.

Por tanto, el modelo no debe interpretarse como una predicción causal, sino como un sistema de clasificación automatizada que aprende una lógica de priorización previamente definida.

En términos de negocio, esto permite aplicar la misma lógica de priorización a nuevas unidades operativas sin tener que recalcular manualmente todo el sistema de scoring.

El modelo no predice una verdad externa independiente, sino que aprende una lógica de priorización construida a partir de reglas de negocio. Su valor está en automatizar esa clasificación para nuevas unidades operativas.

## 3.7 Interpretación de variables del modelo

Después de seleccionar el modelo ganador, se analizarán las variables que más influyen en la clasificación de prioridad de automatización.

Como el modelo ganador es una Regresión Logística, se pueden revisar sus coeficientes para entender qué variables empujan la predicción hacia una clase u otra.

Este análisis ayuda a conectar el modelo con la lógica de negocio del proyecto.

In [ ]:
# Obtener nombres de variables después del preprocesamiento
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()

feature_names[:20]

In [ ]:
# Extraer el modelo interno de regresión logística
logistic_model = best_model.named_steps["model"]

# Crear DataFrame de coeficientes por clase
coefficients_df = pd.DataFrame(
    logistic_model.coef_,
    columns=feature_names,
    index=logistic_model.classes_
)

coefficients_df.head()

In [ ]:
# Mostrar variables con mayor peso positivo por cada clase
top_features_by_class = {}

for class_name in coefficients_df.index:
    top_features = (
        coefficients_df
        .loc[class_name]
        .sort_values(ascending=False)
        .head(15)
        .reset_index()
    )
    
    top_features.columns = ["feature", "coefficient"]
    top_features_by_class[class_name] = top_features
    
    print("\n" + "=" * 80)
    print(f"CLASE: {class_name}")
    print("=" * 80)
    display(top_features)

In [ ]:
# Variables más influyentes para prioridad Alta:
top_features_alta = (
    coefficients_df
    .loc["Alta"]
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

top_features_alta.columns = ["feature", "coefficient"]

top_features_alta

### Interpretación de las variables más influyentes para prioridad alta

Las variables que más impulsan la predicción de prioridad alta están relacionadas principalmente con volumen, tiempos, riesgo operativo y complejidad.

El modelo asigna mayor peso a variables como `total_cases`, `avg_resolution_time_hours`, `sla_breach_rate`, `avg_complexity_score`, `avg_first_response_time_hours` y `escalation_rate`.

Esto es coherente con la lógica de negocio del proyecto: una unidad operativa debe priorizarse para automatización cuando combina alto volumen, tiempos elevados, incumplimiento de SLA, escalaciones frecuentes y mayor complejidad.

También aparecen algunas variables categóricas relacionadas con sistemas, procesos, regiones y canales internos. Sin embargo, las variables numéricas son las que tienen mayor peso, lo que indica que el modelo está priorizando principalmente señales operativas y no solo etiquetas descriptivas.

En términos de negocio, el modelo confirma que las mejores oportunidades de automatización no dependen únicamente del tipo de proceso HR, sino del nivel de fricción operativa que presenta cada unidad.

In [ ]:
# Gráfico de variables para prioridad Alta
top_features_alta_plot = top_features_alta.sort_values("coefficient", ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(
    top_features_alta_plot["feature"],
    top_features_alta_plot["coefficient"]
)

plt.title("Variables que más impulsan la predicción de prioridad Alta")
plt.xlabel("Coeficiente del modelo")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

### Interpretación de variables

El análisis de coeficientes permite identificar qué variables aumentan la probabilidad de que una unidad operativa sea clasificada con prioridad alta, media o baja.

En este proyecto, las variables más importantes deberían estar relacionadas con factores como:

- volumen de casos;
- incumplimiento de SLA;
- escalaciones;
- tiempos de resolución;
- complejidad;
- prioridad alta o urgente;
- tipo de proceso;
- sistema HR;
- canal interno.

Esto es coherente con la lógica de negocio utilizada para construir el Automation Opportunity Score.

### Interpretación de las variables más influyentes para prioridad alta

Las variables que más impulsan la predicción de prioridad alta están relacionadas principalmente con volumen, tiempos, riesgo operativo y complejidad.

El modelo asigna mayor peso a variables como `total_cases`, `avg_resolution_time_hours`, `sla_breach_rate`, `avg_complexity_score`, `avg_first_response_time_hours` y `escalation_rate`.

Esto es coherente con la lógica de negocio del proyecto: una unidad operativa debe priorizarse para automatización cuando combina alto volumen, tiempos elevados, incumplimiento de SLA, escalaciones frecuentes y mayor complejidad.

También aparecen algunas variables categóricas relacionadas con sistemas, procesos, regiones y canales internos. Sin embargo, las variables numéricas son las que tienen mayor peso, lo que indica que el modelo está priorizando principalmente señales operativas y no solo etiquetas descriptivas.

En términos de negocio, el modelo confirma que las mejores oportunidades de automatización no dependen únicamente del tipo de proceso HR, sino del nivel de fricción operativa que presenta cada unidad.

## 3.8 Exportación del modelo y resultados

En esta sección se guardan los principales artefactos del modelo de Machine Learning.

Estos archivos serán utilizados posteriormente para documentación, presentación ejecutiva y construcción de la aplicación en Streamlit.

## Guardar modelo

In [ ]:
import joblib

In [ ]:
model_path = MODELS_PATH / "automation_priority_model.pkl"

joblib.dump(best_model, model_path)

print("Modelo guardado correctamente en:")
print(model_path)

In [ ]:
# Guardar métricas finales en CSV:
model_results_df.to_csv(
    REPORTS_PATH / "model_comparison_results.csv",
    index=False
)

final_model_metrics.to_csv(
    REPORTS_PATH / "final_model_metrics.csv",
    index=False
)

top_features_alta.to_csv(
    REPORTS_PATH / "top_features_priority_alta.csv",
    index=False
)

print("Resultados del modelo guardados correctamente en reports.")


In [ ]:
# Verificar archivos guardados
print("Archivos en models:")
for file_path in MODELS_PATH.glob("*"):
    print(file_path.name)

print("\nArchivos en reports:")
for file_path in REPORTS_PATH.glob("*.csv"):
    print(file_path.name)

## 3.9 Conclusiones del modelo de Machine Learning

En este notebook se construyó un modelo de Machine Learning para predecir la prioridad de automatización de una unidad operativa de HR.

El problema fue definido como una clasificación multiclase, donde la variable objetivo fue `automation_priority`, con tres posibles clases:

- Alta;
- Media;
- Baja.

Se utilizaron variables operativas, categóricas y de rendimiento, incluyendo proceso HR, sistema HR, región, canal interno, volumen de casos, tiempos de resolución, incumplimiento de SLA, escalaciones, complejidad, satisfacción, recurrencia y prioridad de casos.

Se probaron varios modelos de clasificación:

- Logistic Regression;
- Decision Tree;
- Random Forest;
- Gradient Boosting.

El modelo con mejor rendimiento fue Logistic Regression, con un F1 macro aproximado de 0.9685.

Este resultado indica que el modelo fue capaz de aprender con alta precisión la lógica de priorización construida previamente mediante reglas de negocio.

Es importante aclarar que el modelo no predice una verdad externa independiente, sino que aprende una lógica de decisión diseñada para clasificar oportunidades de automatización de forma consistente.

Desde una perspectiva de negocio, el modelo permite automatizar la clasificación de nuevas unidades operativas y mantener un criterio homogéneo para decidir qué procesos deberían priorizarse.

El análisis de variables mostró que las señales más influyentes para predecir prioridad alta fueron:

- volumen de casos;
- tiempo medio de resolución;
- tasa de incumplimiento de SLA;
- complejidad media;
- tiempo de primera respuesta;
- tasa de escalación;
- proporción de casos urgentes;
- proporción de casos de prioridad alta;
- recurrencia de casos.

Esto confirma que el modelo está alineado con la lógica operativa del proyecto: las mejores oportunidades de automatización son aquellas que combinan volumen, fricción, riesgo, complejidad y criticidad.

Finalmente, se guardaron los principales artefactos del modelo:

- modelo entrenado;
- comparación de modelos;
- métricas finales;
- variables más influyentes para prioridad alta.

Estos archivos serán utilizados en la siguiente fase del proyecto: construcción de la aplicación interactiva en Streamlit.